# MMS data analysis

Use pySPEDAS to download MMS magnetic-field and FPI plasma-moment data, inspect the products returned for a chosen interval, and make a quick-look plot. The notebook starts with burst data when it is available and otherwise uses fast survey data.

## Requirements

Install ShockGeo with its optional MMS dependencies before running this notebook:

```bash
pip install -e ".[mms]"
```

The first download may take a little longer while pySPEDAS obtains data files. Internet access is required.

In [ ]:
from pathlib import Path
import sys

# Support launching Jupyter from either the repository root or examples/.
EXAMPLES_DIRECTORY = Path.cwd() / "examples"
if not (EXAMPLES_DIRECTORY / "mms_data_analysis.py").is_file():
    EXAMPLES_DIRECTORY = Path.cwd()
sys.path.insert(0, str(EXAMPLES_DIRECTORY.resolve()))

from mms_data_analysis import load_mms_data, plot_mms_data, summarize_data


## Parameters

Edit this cell to select the spacecraft and time interval. Set `MODE` to `"auto"` to prefer burst data and fall back to fast data, `"brst"` to request burst only, or `"fast"` for survey data only. Short intervals near known burst periods are most likely to return burst data.

In [ ]:
# Edit these values for the MMS interval you want to analyze.
START = "2015-10-16 13:06:00"
END = "2015-10-16 13:07:00"
PROBE = 1
MODE = "auto"  # auto prefers burst, then falls back to fast


## Download MMS data

This loads FGM magnetic-field data and FPI ion/electron density, velocity, and temperature moments. The status line reports the cadence actually used.

In [ ]:
data = load_mms_data(START, END, probe=PROBE, mode=MODE)
if not data.series:
    raise RuntimeError(
        "No MMS data found; try MODE = 'fast', a shorter interval, or another time."
    )
print(f"Loaded MMS{PROBE} {data.cadence} data with {len(data.series)} products.")


## Inspect loaded products

Review which physical quantities were available for this interval. Availability can vary by cadence, spacecraft, and instrument mode.

In [ ]:
for name, series in data.series.items():
    print(f"{name}: {series.label} [{series.units}]")


## Summary statistics

Compute minimum, mean, and maximum values for each loaded component. Use these values as a quick data-quality check before interpreting the plot.

In [ ]:
summary = summarize_data(data)
summary


## Plot data

Create a multi-panel quick-look figure for magnetic field, density, velocity, and temperature. Missing products are omitted automatically.

In [ ]:
figure = plot_mms_data(data)
figure.show()


## Troubleshooting

- If no data load, try a shorter interval or set `MODE = "fast"`; burst coverage is intermittent.
- If imports fail, rerun `pip install -e ".[mms]"` in the same Python environment as Jupyter, then restart the kernel.
- If a product is absent, inspect the loaded-product list; FPI moments are not guaranteed for every requested interval.